# Document Scanning & Enhancement — CNN Applications
Fundamentals of Computer Vision, Spring 2026.

Two mandatory tasks + bonus, all in this notebook:
1. **Enhancement network** (U-Net style, from scratch) — rectified degraded crop -> clean scan.
2. **Corner detection** — direct coordinate regression (A) vs heatmap regression (B).
3. **Section 6** — dropout-regularized re-training of both.
4. **Bonus** — end-to-end scanner: raw photo -> predicted corners -> warp -> enhance.

Degradations are built **only with OpenCV/NumPy** (no albumentations etc.), and the
synthetic generator produces the (degraded input, clean target, 4 corners) triple in one shot.

In [ ]:
import os
import io
import json
import math
import random
import glob
import time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import multiprocessing as mp

try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SCAN_DIR = 'Sample'
BG_DIR = 'Backgrounds'
DATA_DIRS = {
    'train': 'TrainingData',
    'val': 'ValidationData',
    'test': 'TestData'
}
N_PER_SCAN = {
    'train': 24,
    'val': 8,
    'test': 8
}
REGEN = False
REAL_DIR = 'real_photos'
REAL_JSON = 'real_photos/_annotations.coco.json'
REF_DIR = 'real_photos/reference'

IMG = 256
PHOTO = 256
OUT = 'out'
os.makedirs(OUT, exist_ok=True)
NW = 0

PRETRAINED = {
    'enh': 'enhancement_model_weights.pth',
    'reg': 'out/reg_best.pt',
    'hm': 'out/hm_best.pt'
}
FORCE_TRAIN = False

## 1. Dataset

### 1.1 Provided clean scans
Split **by source scan** 80/10/10 so no page ever appears on two sides of a split.

In [ ]:
scans = sorted(f for f in glob.glob(os.path.join(SCAN_DIR, '*')) if os.path.isfile(f))
assert scans, f'no scans found in {SCAN_DIR}'
print(len(scans), 'clean scans')

rng = random.Random(SEED)
order = scans[:]
rng.shuffle(order)

n = len(order)
n_tr = int(0.8 * n)
n_va = max(1, int(0.1 * n))

SPLIT = {
    'train': order[:n_tr],
    'val': order[n_tr:n_tr + n_va],
    'test': order[n_tr + n_va:]
}

{k: len(v) for k, v in SPLIT.items()}

In [ ]:
def load_scan(path, long_side=768):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    s = long_side / max(h, w)
    if s < 1:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return img

SCAN_CACHE = {p: load_scan(p) for p in scans}

plt.figure(figsize=(9, 3))
for i, p in enumerate(scans[:3]):
    plt.subplot(1, 3, i + 1)
    plt.imshow(cv2.cvtColor(SCAN_CACHE[p], cv2.COLOR_BGR2RGB))
    plt.axis('off')
plt.suptitle('clean scans (targets)')
plt.show()

### 1.3 Synthetic generation + Section 4 degradation pipeline

No backgrounds were shipped with the dataset, so backgrounds are **procedurally synthesized**
(wood grain, fabric/carpet noise, flat desk, clutter) with OpenCV primitives. Drop real
background photos into `backgrounds/` and they are used instead — the generator picks them up
automatically.

Pipeline for one sample (Section 4.3 order):
1. random perspective warp of the clean scan onto a background (**the 4 target points are the labels**)
2. random downscale/upscale 2x–4x
3. brightness / contrast / warm-cool color cast
4. smooth illumination gradient + soft blurred-polygon shadows
5. Gaussian (or slight motion) blur + Gaussian noise
6. JPEG re-encode at quality 30–80

The **input for the enhancement net** is that degraded photo warped *back* with the recorded
homography, so it is pixel-aligned with the clean target. Photometric damage is applied to the
input only.

In [ ]:
BG_FILES = sorted(f for f in glob.glob(os.path.join(BG_DIR, '*')) if os.path.isfile(f))
print(len(BG_FILES), 'background photos')


def synth_background(h, w, rs):
    kind = rs.randint(0, 4)
    base = np.zeros((h, w, 3), np.float32)

    if kind == 0:
        col = np.array([60, 105, 150], np.float32) + rs.uniform(-25, 25, 3)
        base[:] = col
        grain = rs.normal(0, 1, (h, w)).astype(np.float32)
        grain = cv2.GaussianBlur(grain, (0, 0), (1.0, 25.0)[rs.randint(0, 2)])
        base += (grain / (np.abs(grain).max() + 1e-6) * 25)[..., None]

    elif kind == 1:
        col = np.array([90, 90, 100], np.float32) + rs.uniform(-40, 60, 3)
        noise = rs.normal(0, 18, (h, w, 3)).astype(np.float32)
        base = col + cv2.GaussianBlur(noise, (0, 0), 1.2)

    elif kind == 2:
        base[:] = np.array([120, 120, 120], np.float32) + rs.uniform(-60, 80, 3)

    else:
        base[:] = np.array([100, 100, 110], np.float32) + rs.uniform(-40, 60, 3)
        for _ in range(rs.randint(3, 9)):
            p0 = (rs.randint(0, w), rs.randint(0, h))
            p1 = (rs.randint(0, w), rs.randint(0, h))
            c = tuple(float(v) for v in rs.uniform(0, 255, 3))
            if rs.rand() < 0.5:
                cv2.rectangle(base, p0, p1, c, -1)
            else:
                cv2.circle(base, p0, rs.randint(10, 90), c, -1)
        base = cv2.GaussianBlur(base, (0, 0), 2.0)

    g = cv2.GaussianBlur(rs.normal(0, 1, (h, w)).astype(np.float32), (0, 0), max(h, w) / 6)
    base *= (1 + 0.25 * g / (np.abs(g).max() + 1e-6))[..., None]

    return np.clip(base, 0, 255).astype(np.uint8)


def get_background(h, w, rs):
    if BG_FILES:
        bg = cv2.imread(BG_FILES[rs.randint(0, len(BG_FILES))])
        bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_AREA)
        return bg
    return synth_background(h, w, rs)

In [ ]:
def deg_rescale(img, rs):
    f = rs.uniform(2.0, 4.0)
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(8, int(w / f)), max(8, int(h / f))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)


def deg_photometric(img, rs):
    x = img.astype(np.float32)
    a = rs.uniform(0.8, 1.2)
    b = rs.uniform(-20, 20)
    hi = np.percentile(x, 99)
    b = min(b, 252 - a * hi)
    b = max(b, 205 - a * hi)
    x = x * a + b
    cast = np.array([rs.uniform(0.9, 1.1), 1.0, rs.uniform(0.9, 1.1)], np.float32)
    return np.clip(x * cast, 0, 255)


def deg_illumination(img, rs):
    h, w = img.shape[:2]
    out = img.astype(np.float32)

    for _ in range(rs.randint(1, 5)):
        s = 256
        mask = np.zeros((s, s), np.float32)
        k = rs.randint(3, 7)
        pts = np.stack([rs.uniform(-0.2, 1.2, k) * s, rs.uniform(-0.2, 1.2, k) * s], 1).astype(np.int32)
        cv2.fillConvexPoly(mask, cv2.convexHull(pts), 1.0)

        hard = rs.rand() < 0.45
        sigma = rs.uniform(0.004, 0.02) if hard else rs.uniform(0.06, 0.20)
        mask = cv2.GaussianBlur(mask, (0, 0), sigma * s)
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)

        if mask.mean() > 0.7:
            continue

        d = rs.uniform(0.25, 0.60) if hard else rs.uniform(0.15, 0.45)
        tint = np.float32([1.0, rs.uniform(1.02, 1.10), rs.uniform(1.05, 1.20)])
        out *= (1 - d * mask[..., None] * tint)

    return np.clip(out, 0, 255)


def deg_blur_noise(img, rs):
    x = img.astype(np.float32)

    if rs.rand() < 0.35:
        L = rs.randint(3, 10)
        k = np.zeros((L, L), np.float32)
        cv2.line(k, (0, L // 2), (L - 1, L // 2), 1.0, 1)
        M = cv2.getRotationMatrix2D((L / 2 - 0.5, L / 2 - 0.5), rs.uniform(0, 180), 1)
        k = cv2.warpAffine(k, M, (L, L))
        k /= k.sum() + 1e-8
        x = cv2.filter2D(x, -1, k)
    else:
        x = cv2.GaussianBlur(x, (0, 0), rs.uniform(0.4, 1.8))

    x += rs.normal(0, rs.uniform(2, 9), x.shape)
    return np.clip(x, 0, 255)


def deg_jpeg(img, rs):
    q = int(rs.randint(30, 81))
    ok, buf = cv2.imencode('.jpg', np.clip(img, 0, 255).astype(np.uint8),
                           [int(cv2.IMWRITE_JPEG_QUALITY), q])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

In [ ]:
MAX_F = 4.0
HEAD = 1.5


def supersample(page_px, crop_size):
    return int(np.clip(np.ceil(MAX_F * HEAD * crop_size / max(page_px, 1)), 2, 12))


def make_sample(scan, rs, photo_size=PHOTO, crop_size=IMG):
    h, w = scan.shape[:2]
    src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])

    m = rs.uniform(0.03, 0.16)
    sx, sy = rs.uniform(0.55, 0.95), rs.uniform(0.55, 0.95)
    cxo, cyo = rs.uniform(-0.06, 0.06), rs.uniform(-0.06, 0.06)

    SS = supersample(sx * photo_size * (1 - 2 * m), crop_size)
    H = W = photo_size * SS

    bw, bh = sx * W * (1 - 2 * m), sy * H * (1 - 2 * m)
    cx, cy = W / 2 + cxo * W, H / 2 + cyo * H

    base = np.float32([
        [cx - bw / 2, cy - bh / 2],
        [cx + bw / 2, cy - bh / 2],
        [cx + bw / 2, cy + bh / 2],
        [cx - bw / 2, cy + bh / 2]
    ])

    jit = rs.uniform(-0.13, 0.13, (4, 2)) * np.float32([bw, bh])
    ang = rs.uniform(-0.25, 0.25)
    R = np.float32([
        [np.cos(ang), -np.sin(ang)],
        [np.sin(ang), np.cos(ang)]
    ])

    dst = ((base + jit) - [cx, cy]) @ R.T + [cx, cy]
    dst = np.clip(dst, 2, [W - 3, H - 3]).astype(np.float32)

    Hm = cv2.getPerspectiveTransform(src, dst)
    bg = get_background(H, W, rs)

    page = cv2.warpPerspective(scan, Hm, (W, H), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

    mask = cv2.warpPerspective(np.full((h, w), 255, np.uint8), Hm, (W, W))
    mask = cv2.GaussianBlur(mask, (0, 0), 0.8)[..., None].astype(np.float32) / 255.

    photo = (page.astype(np.float32) * mask + bg.astype(np.float32) * (1 - mask))
    photo = np.clip(photo, 0, 255).astype(np.uint8)

    for fn in (deg_rescale, deg_photometric, deg_illumination, deg_blur_noise, deg_jpeg):
        photo = fn(photo, rs)

    photo = np.clip(photo, 0, 255).astype(np.uint8)

    big = crop_size * SS
    rect = np.float32([
        [0, 0],
        [big - 1, 0],
        [big - 1, big - 1],
        [0, big - 1]
    ])

    Hb = cv2.getPerspectiveTransform(dst, rect)
    inp = cv2.warpPerspective(photo, Hb, (big, big), flags=cv2.INTER_LINEAR)
    inp = cv2.resize(inp, (crop_size, crop_size), interpolation=cv2.INTER_AREA)

    photo = cv2.resize(np.clip(photo, 0, 255).astype(np.uint8), (photo_size, photo_size),
                       interpolation=cv2.INTER_AREA)
    dst = dst / SS
    tgt = cv2.resize(scan, (crop_size, crop_size), interpolation=cv2.INTER_AREA)

    return dict(photo=photo, corners=dst.copy(), inp=inp, tgt=tgt)

### 2.4 / 4.4 Verification — corners overlaid, and the input/target alignment

In [ ]:
rs = np.random.RandomState(0)

fig, ax = plt.subplots(3, 3, figsize=(10, 10))

for r in range(3):
    s = make_sample(SCAN_CACHE[SPLIT['train'][r]], rs)

    vis = s['photo'].copy()

    for i, (x, y) in enumerate(s['corners']):
        cv2.circle(vis, (int(x), int(y)), 5, (0, 0, 255), -1)
        cv2.putText(vis, 'TL TR BR BL'.split()[i], (int(x) + 6, int(y)), 0, 0.45, (0, 255, 255), 1)

    cv2.polylines(vis, [s['corners'].astype(np.int32)], True, (0, 255, 0), 2)

    for c, (im, t) in enumerate(zip([vis, s['inp'], s['tgt']],
                                    ['composited photo + labels',
                                     'degraded rectified input',
                                     'clean target'])):
        ax[r, c].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax[r, c].set_title(t, fontsize=9)
        ax[r, c].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
s = make_sample(SCAN_CACHE[SPLIT['train'][0]], np.random.RandomState(3))

def edge_detect(im):
    return cv2.Canny(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), 60, 160)

edges_inp = edge_detect(s['inp'])
edges_tgt = edge_detect(s['tgt'])

overlay = np.dstack([edges_inp, edges_tgt, np.zeros_like(edges_tgt)])

plt.figure(figsize=(4, 4))
plt.imshow(overlay)
plt.title('red=input edges, green=target edges')
plt.axis('off')
plt.show()

## 2. Datasets & loaders

The generator above is run **once** to materialize the dataset on disk, in the folder layout that
was already in the project:

```
TrainingData/ ValidationData/ TestData/
  Task1_Enhancement/input/img_<scan>_<k>.jpg     degraded, rectified
  Task1_Enhancement/target/img_<scan>_<k>.jpg    clean scan
  Task2_CornerDetection/images/img_<scan>_<k>.jpg   composited photo
  Task2_CornerDetection/corners.json             {filename: [[x,y] x4]}  TL TR BR BL, pixels
```

Writing it once freezes **all three** splits — validation and test are scored on identical images
for every model, and training is reproducible. Set `REGEN = False` to reuse what is on disk.

In [ ]:
SPLIT_SALT = {'train': 0, 'val': 1, 'test': 2}


def split_dirs(split):
    r = DATA_DIRS[split]
    return (os.path.join(r, 'Task1_Enhancement', 'input'),
            os.path.join(r, 'Task1_Enhancement', 'target'),
            os.path.join(r, 'Task2_CornerDetection', 'images'),
            os.path.join(r, 'Task2_CornerDetection', 'corners.json'))


def build_split(split):
    din, dtg, dph, jf = split_dirs(split)

    for d in (din, dtg, dph):
        os.makedirs(d, exist_ok=True)

    if not REGEN and os.path.exists(jf) and json.load(open(jf)):
        print(f"Using cached {split} data")
        return json.load(open(jf))

    for d in (din, dtg, dph):
        for f in glob.glob(os.path.join(d, '*.jpg')):
            os.remove(f)

    corners = {}
    t0 = time.time()

    for si, path in enumerate(SPLIT[split]):
        scan = SCAN_CACHE[path]

        for k in range(N_PER_SCAN[split]):
            rs = np.random.RandomState(SEED + 100000 * SPLIT_SALT[split] + 1000 * si + k)
            s = make_sample(scan, rs, photo_size=512)

            name = f'img_{si + 1}_{k + 1}.jpg'

            cv2.imwrite(os.path.join(din, name), s['inp'], [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(os.path.join(dtg, name), s['tgt'], [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(os.path.join(dph, name), s['photo'], [cv2.IMWRITE_JPEG_QUALITY, 95])

            corners[name] = s['corners'].tolist()

    json.dump(corners, open(jf, 'w'))
    print(f'{split:6s}{len(corners):5d} samples  ({time.time() - t0:.1f}s)  -> {DATA_DIRS[split]}')
    return corners


CORNERS = {sp: build_split(sp) for sp in ('train', 'val', 'test')}
{k: len(v) for k, v in CORNERS.items()}

In [ ]:
def to_chw(img):
    return torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1).copy()).float() / 255.0


def make_heatmaps(corners_norm, size, sigma):
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    hm = np.stack([
        np.exp(-(((xx - cx * size) ** 2 + (yy - cy * size) ** 2) / (2 * sigma ** 2)))
        for cx, cy in corners_norm
    ]).astype(np.float32)
    return torch.from_numpy(hm)


class DiskDataset(Dataset):
    def __init__(self, split, mode='enhance', hm_size=64, sigma=2.0):
        self.din, self.dtg, self.dph, _ = split_dirs(split)
        self.names = sorted(CORNERS[split].keys())
        self.corners = CORNERS[split]
        self.mode = mode
        self.hm_size = hm_size
        self.sigma = sigma

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        n = self.names[i]

        if self.mode == 'enhance':
            x = cv2.resize(cv2.imread(os.path.join(self.din, n)), (IMG, IMG))
            y = cv2.resize(cv2.imread(os.path.join(self.dtg, n)), (IMG, IMG))
            return to_chw(x), to_chw(y)

        ph = cv2.imread(os.path.join(self.dph, n))
        h, w = ph.shape[:2]
        c = np.asarray(self.corners[n], np.float32) / np.float32([w, h])
        ph = cv2.resize(ph, (PHOTO, PHOTO))
        return to_chw(ph), torch.from_numpy(c).float(), make_heatmaps(c, self.hm_size, self.sigma)


BS = 16
MPC = 'fork' if NW > 0 else None


def make_loader(split, mode, shuffle=False):
    return DataLoader(
        DiskDataset(split, mode),
        BS,
        shuffle=shuffle,
        num_workers=NW,
        persistent_workers=NW > 0,
        multiprocessing_context=MPC,
        drop_last=shuffle
    )


enh_tr = make_loader('train', 'enhance', True)
enh_va = make_loader('val', 'enhance')
enh_te = make_loader('test', 'enhance')
enh_tr_eval = make_loader('train', 'enhance')

cor_tr = make_loader('train', 'corner', True)
cor_va = make_loader('val', 'corner')
cor_te = make_loader('test', 'corner')

x, y = next(iter(enh_va))
print(x.shape, y.shape, x.min().item(), x.max().item())

## 3. Task 1 — Enhancement network

Encoder–decoder with skip connections, written from scratch (no torchvision, no ImageNet weights).
Layer names and shapes match the checkpoints already in the project folder
(`enhancement_model_weights.pth`, `corner_detector_*_weights.pth`), so those can be loaded instead
of retraining — set `FORCE_TRAIN = True` to ignore them. `p_drop=0` for Section 3; Section 6
re-instantiates with dropout.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout, p=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.BatchNorm2d(cout),
            nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.BatchNorm2d(cout),
            nn.ReLU(True),
            nn.Dropout2d(p)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, cin=3, cout=3, base=32, p_drop=0.0, out_act='sigmoid'):
        super().__init__()
        b = base

        self.down1 = DoubleConv(cin, b)
        self.down2 = DoubleConv(b, b * 2)
        self.down3 = DoubleConv(b * 2, b * 4)
        self.bottleneck = DoubleConv(b * 4, b * 8, p_drop)

        self.pool = nn.MaxPool2d(2)

        self.up1 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2)
        self.up_conv1 = DoubleConv(b * 8, b * 4)

        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2)
        self.up_conv2 = DoubleConv(b * 4, b * 2)

        self.up3 = nn.ConvTranspose2d(b * 2, b, 2, 2)
        self.up_conv3 = DoubleConv(b * 2, b)

        self.final_conv = nn.Conv2d(b, cout, 1)
        self.out_act = out_act

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))

        z = self.bottleneck(self.pool(d3))

        u = self.up_conv1(torch.cat([self.up1(z), d3], 1))
        u = self.up_conv2(torch.cat([self.up2(u), d2], 1))
        u = self.up_conv3(torch.cat([self.up3(u), d1], 1))

        y = self.final_conv(u)

        if self.out_act == 'sigmoid':
            return torch.sigmoid(y)
        return y


def load_pretrained(model, path, remap=None):
    if FORCE_TRAIN or not os.path.exists(path):
        return False

    sd = torch.load(path, map_location='cpu')

    if isinstance(sd, dict) and 'state_dict' in sd:
        sd = sd['state_dict']

    if remap:
        sd = {remap.get(k, k): v for k, v in sd.items()}

    missing, unexpected = model.load_state_dict(sd, strict=False)

    ok = not unexpected and not [k for k in missing if 'num_batches' not in k]

    print(('loaded  ' if ok else 'PARTIAL ') + path,
          '' if ok else f'(missing {len(missing)}, unexpected {len(unexpected)})')

    return ok


print(sum(p.numel() for p in UNet().parameters()) / 1e6, 'M params')

### 3.2 Loss

MSE gives blurry text, so the loss is **L1 + (1 - SSIM) + L1 on Sobel gradients** — pixel
fidelity, structure, and edge sharpness. SSIM is implemented here (used for the metric too).

In [ ]:
def _gauss(ws=11, sigma=1.5, ch=3, dev='cpu'):
    g = torch.arange(ws).float() - ws // 2
    g = torch.exp(-(g ** 2) / (2 * sigma ** 2))
    g /= g.sum()
    k = (g[:, None] @ g[None, :]).expand(ch, 1, ws, ws).contiguous()
    return k.to(dev)


def ssim(a, b, ws=11, C1=0.01 ** 2, C2=0.03 ** 2):
    ch = a.shape[1]
    k = _gauss(ws, 1.5, ch, a.device)
    pad = ws // 2

    mu1 = F.conv2d(a, k, padding=pad, groups=ch)
    mu2 = F.conv2d(b, k, padding=pad, groups=ch)

    s11 = F.conv2d(a * a, k, padding=pad, groups=ch) - mu1 ** 2
    s22 = F.conv2d(b * b, k, padding=pad, groups=ch) - mu2 ** 2
    s12 = F.conv2d(a * b, k, padding=pad, groups=ch) - mu1 * mu2

    numerator = (2 * mu1 * mu2 + C1) * (2 * s12 + C2)
    denominator = (mu1 ** 2 + mu2 ** 2 + C1) * (s11 + s22 + C2)
    m = numerator / denominator

    return m.mean(dim=[1, 2, 3])


SOBX = torch.tensor([[1., 0, -1], [2, 0, -2], [1, 0, -1]]).view(1, 1, 3, 3)
SOBY = SOBX.transpose(2, 3).contiguous()


def sobel(x):
    g = x.mean(1, keepdim=True)
    kx, ky = SOBX.to(x.device), SOBY.to(x.device)
    return torch.cat([F.conv2d(g, kx, padding=1), F.conv2d(g, ky, padding=1)], 1)


def chroma(x):
    return x - x.mean(1, keepdim=True)


def enh_loss(pred, tgt, w=(1.0, 0.8, 1.0, 0.6)):
    l1 = F.l1_loss(pred, tgt)
    lssm = (1 - ssim(pred, tgt)).mean()
    ledg = F.l1_loss(sobel(pred), sobel(tgt))
    lchr = F.l1_loss(chroma(pred), chroma(tgt))

    loss = w[0] * l1 + w[1] * lssm + w[2] * ledg + w[3] * lchr

    components = {
        'l1': l1.item(),
        'ssim': lssm.item(),
        'edge': ledg.item(),
        'chroma': lchr.item()
    }

    return loss, components


def psnr(a, b):
    mse = ((a - b) ** 2).mean(dim=[1, 2, 3]).clamp_min(1e-10)
    return 10 * torch.log10(1.0 / mse)

In [ ]:
def train_enhance(model, epochs=25, lr=2e-3, tag='enh'):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs * len(enh_tr))
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    hist = {'train': [], 'val': []}
    best = 1e9

    for ep in range(epochs):
        model.train()
        tl = 0

        for x, y in enh_tr:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss, _ = enh_loss(model(x), y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

            tl += loss.item()

        model.eval()
        vl = 0

        with torch.no_grad():
            for x, y in enh_va:
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                vl += enh_loss(model(x), y)[0].item()

        tl /= len(enh_tr)
        vl /= len(enh_va)

        hist['train'].append(tl)
        hist['val'].append(vl)

        if vl < best:
            best = vl
            torch.save(model.state_dict(), f'{OUT}/{tag}_best.pt')

        print(f'ep {ep + 1:02d}/{epochs}  train {tl:.4f}  val {vl:.4f}{"  *" if vl == best else ""}')

    model.load_state_dict(torch.load(f'{OUT}/{tag}_best.pt'))
    return model, hist


EPOCHS = 25
enh_model = UNet(p_drop=0.0).to(DEVICE)
enh_hist = {'train': [], 'val': []}

if not load_pretrained(enh_model, PRETRAINED['enh']):
    enh_model, enh_hist = train_enhance(UNet(p_drop=0.0), epochs=EPOCHS, tag='enh')

In [ ]:
if not enh_hist['train']:
    print('loaded a checkpoint — no curve to plot (set FORCE_TRAIN=True to train)')

plt.figure(figsize=(5, 3.4))
plt.plot(enh_hist['train'], label='train')
plt.plot(enh_hist['val'], label='val')
plt.xlabel('epoch')
plt.ylabel('L1+SSIM+edge loss')
plt.legend()
plt.title('enhancement training')
plt.grid(alpha=0.3)
plt.show()

### 3.3 Evaluation — PSNR / SSIM table

Row 0 is the **do-nothing baseline**: the metrics of the degraded input itself on the test bucket.

In [ ]:
@torch.no_grad()
def eval_enh(model, loader):
    model.eval()
    P = S = Pb = Sb = k = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        p = model(x).clamp(0, 1)

        P += psnr(p, y).sum().item()
        S += ssim(p, y).sum().item()
        Pb += psnr(x, y).sum().item()
        Sb += ssim(x, y).sum().item()
        k += x.size(0)

    return P / k, S / k, Pb / k, Sb / k


rows = []

for name, ld in [('Training', enh_tr_eval), ('Validation', enh_va), ('Test', enh_te)]:
    p, s, pb, sb = eval_enh(enh_model, ld)
    rows.append((name, p, s, pb, sb))

print(f"{'Split':12s}{'PSNR':>9s}{'SSIM':>9s}   | baseline(input) PSNR/SSIM")
print(f"{'Baseline*':12s}{rows[2][3]:9.2f}{rows[2][4]:9.4f}   <- degraded input, test bucket")

for n, p, s, pb, sb in rows:
    print(f'{n:12s}{p:9.2f}{s:9.4f}   |{pb:9.2f}{sb:9.4f}')

ENH_TABLE = rows

In [ ]:
@torch.no_grad()
def show_enh(model, loader, k=3):
    model.eval()
    x, y = next(iter(loader))
    x = x[:k].to(DEVICE)
    y = y[:k].to(DEVICE)
    p = model(x).clamp(0, 1).cpu()

    fig, ax = plt.subplots(k, 3, figsize=(9, 3 * k))

    for i in range(k):
        for j, (im, t) in enumerate(zip(
            [x[i].cpu(), p[i], y[i].cpu()],
            ['degraded input', 'our output', 'clean target']
        )):
            ax[i, j].imshow(im.permute(1, 2, 0).numpy())
            ax[i, j].set_title(t, fontsize=9)
            ax[i, j].axis('off')

    plt.tight_layout()
    plt.show()


show_enh(enh_model, enh_te)

### 3.4 Enhancement inference pipeline
Takes an **unseen rectified document image** (BGR uint8, any size) and returns a clean uint8 image
at the original resolution.

In [ ]:
@torch.no_grad()
def enhance_pipeline(rect_bgr, model=None, show=False):
    model = (model or enh_model).eval()
    h, w = rect_bgr.shape[:2]

    x = to_chw(cv2.resize(rect_bgr, (IMG, IMG), interpolation=cv2.INTER_AREA))
    x = x[None].to(DEVICE)

    y = model(x).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0)
    out = cv2.cvtColor((y * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    out = cv2.resize(out, (w, h), interpolation=cv2.INTER_CUBIC)

    if show:
        plt.figure(figsize=(8, 4))
        for i, (im, t) in enumerate([(rect_bgr, 'input'), (out, 'enhanced')]):
            plt.subplot(1, 2, i + 1)
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            plt.title(t)
            plt.axis('off')
        plt.show()

    return out


s = make_sample(SCAN_CACHE[SPLIT['test'][0]], np.random.RandomState(11))
_ = enhance_pipeline(s['inp'], show=True)

## 5. Task 2 — Corner detection

**Prediction before running:** heatmaps should win on accuracy. Direct regression asks fully
connected layers to squeeze a global feature vector into precise coordinates, discarding spatial
structure; heatmap regression keeps the problem spatial and local, so each corner is localized by
evidence right where it lies. Regression should train faster and be more stable when the page is
partly out of frame (a heatmap has no peak to place there).

### Approach A — direct coordinate regression

In [ ]:
class CornerRegressor(nn.Module):
    def __init__(self, p_drop=0.0):
        super().__init__()
        L = []
        cin = 3

        for c in (32, 64, 128, 256, 512):
            L.append(nn.Conv2d(cin, c, 3, padding=1))
            L.append(nn.ReLU(True))
            L.append(nn.MaxPool2d(2))
            cin = c

        self.features = nn.Sequential(*L)

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 8 * 8, 512),
            nn.ReLU(True),
            nn.Dropout(p_drop),
            nn.Linear(512, 8),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.view(-1, 4, 2)


REG_REMAP = {
    'regressor.3.weight': 'regressor.4.weight',
    'regressor.3.bias': 'regressor.4.bias'
}


def soft_argmax(hm):
    B, K, S, _ = hm.shape
    p = F.softmax(hm.view(B, K, -1) * 10, -1).view(B, K, S, S)

    idx = torch.linspace(0, 1, S, device=hm.device)
    x = (p.sum(2) * idx).sum(-1)
    y = (p.sum(3) * idx).sum(-1)

    return torch.stack([x, y], -1)


class CornerHeatmap(UNet):
    def __init__(self, p_drop=0.0):
        super().__init__(3, 4, 32, p_drop, out_act='linear')

    def forward(self, x):
        x = super().forward(x)
        return F.interpolate(x, size=(64, 64), mode='bilinear', align_corners=False)

In [ ]:
def corner_metrics(pred_n, gt_n, px=PHOTO, thr=8.0):
    d = (pred_n - gt_n).mul(px).pow(2).sum(-1).sqrt()
    return d.mean().item(), (d.max(1).values < thr).float().mean().item(), d


def train_corner(model, kind, epochs=25, lr=2e-3, tag='cor'):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs * len(cor_tr))
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    hist = {'train': [], 'val': [], 'err': []}
    best = 1e9

    for ep in range(epochs):
        model.train()
        tl = 0

        for x, c, hm in cor_tr:
            x = x.to(DEVICE, non_blocking=True)
            c = c.to(DEVICE)
            hm = hm.to(DEVICE)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                out = model(x)

                if kind == 'reg':
                    loss = F.l1_loss(out, c)
                else:
                    loss = F.mse_loss(torch.sigmoid(out), hm) + 0.1 * F.l1_loss(soft_argmax(out), c)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

            tl += loss.item()

        model.eval()
        vl = 0
        errs = []

        with torch.no_grad():
            for x, c, hm in cor_va:
                x = x.to(DEVICE)
                c = c.to(DEVICE)
                hm = hm.to(DEVICE)

                out = model(x)

                if kind == 'reg':
                    loss = F.l1_loss(out, c)
                    pred = out
                else:
                    loss = F.mse_loss(torch.sigmoid(out), hm) + 0.1 * F.l1_loss(soft_argmax(out), c)
                    pred = soft_argmax(out)

                vl += loss.item()
                errs.append(corner_metrics(pred, c)[2])

        vl /= len(cor_va)
        err = torch.cat(errs).mean().item()

        hist['train'].append(tl / len(cor_tr))
        hist['val'].append(vl)
        hist['err'].append(err)

        if vl < best:
            best = vl
            torch.save(model.state_dict(), f'{OUT}/{tag}_best.pt')

        print(f'ep {ep + 1:02d}/{epochs}  train {tl / len(cor_tr):.5f}  val {vl:.5f}  mean corner err {err:.2f}px')

    model.load_state_dict(torch.load(f'{OUT}/{tag}_best.pt'))
    return model, hist


reg_model = CornerRegressor().to(DEVICE)
reg_hist = {'err': []}

if not load_pretrained(reg_model, PRETRAINED['reg'], REG_REMAP):
    reg_model, reg_hist = train_corner(CornerRegressor(), 'reg', epochs=EPOCHS, tag='reg')

hm_model = CornerHeatmap().to(DEVICE)
hm_hist = {'err': []}

if not load_pretrained(hm_model, PRETRAINED['hm']):
    hm_model, hm_hist = train_corner(CornerHeatmap(), 'hm', epochs=EPOCHS, tag='hm')

In [ ]:
@torch.no_grad()
def eval_corner(model, kind, loader, thr=8.0):
    model.eval()
    D = []

    for x, c, _ in loader:
        out = model(x.to(DEVICE))
        pred = out if kind == 'reg' else soft_argmax(out)
        D.append(corner_metrics(pred, c.to(DEVICE), thr=thr)[2])

    D = torch.cat(D)
    return D.mean().item(), (D.max(1).values < thr).float().mean().item()


print(f"{'model':12s}{'mean err px':>13s}{'all-4 < 8px':>14s}")

for nm, m, k in [('regression', reg_model, 'reg'), ('heatmap', hm_model, 'hm')]:
    e, a = eval_corner(m, k, cor_te)
    print(f'{nm:12s}{e:13.2f}{a:14.2%}')

plt.figure(figsize=(5, 3.4))
plt.plot(reg_hist['err'], label='regression')
plt.plot(hm_hist['err'], label='heatmap')
plt.xlabel('epoch')
plt.ylabel('mean corner error (px)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
@torch.no_grad()
def show_corners(loader, k=3):
    x, c, _ = next(iter(loader))
    x = x[:k].to(DEVICE)
    c = c[:k]

    pr = reg_model(x).cpu()
    ph = soft_argmax(hm_model(x)).cpu()

    fig, ax = plt.subplots(k, 2, figsize=(7, 3.4 * k))

    for i in range(k):
        im = (x[i].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()

        for j, (p, ttl) in enumerate([(pr[i], 'A: regression'), (ph[i], 'B: heatmap')]):
            v = im.copy()

            cv2.polylines(v, [(c[i].numpy() * PHOTO).astype(np.int32)], True, (0, 255, 0), 2)
            cv2.polylines(v, [(p.numpy() * PHOTO).astype(np.int32)], True, (255, 0, 0), 2)

            ax[i, j].imshow(v)
            ax[i, j].set_title(f'{ttl} (green=GT, red=pred)', fontsize=9)
            ax[i, j].axis('off')

    plt.tight_layout()
    plt.show()


show_corners(cor_te)

### 5.1 Corner inference pipeline
Raw photo in, corners in **original-image pixels** out. Uses whichever model scored better.

In [ ]:
BEST_CORNER = ('hm', hm_model) if eval_corner(hm_model, 'hm', cor_te)[0] < eval_corner(reg_model, 'reg', cor_te)[0] else ('reg', reg_model)
print('best corner model:', BEST_CORNER[0])


@torch.no_grad()
def corner_pipeline(photo_bgr, model=None, kind=None, show=False):
    kind = kind or BEST_CORNER[0]
    model = (model or BEST_CORNER[1]).eval()

    h, w = photo_bgr.shape[:2]

    x = to_chw(cv2.resize(photo_bgr, (PHOTO, PHOTO), interpolation=cv2.INTER_AREA))
    x = x[None].to(DEVICE)

    out = model(x)
    pred = (out if kind == 'reg' else soft_argmax(out))[0].cpu().numpy()
    pts = pred * np.float32([w, h])

    if show:
        v = photo_bgr.copy()
        cv2.polylines(v, [pts.astype(np.int32)], True, (0, 0, 255), max(2, w // 300))

        for i, (px, py) in enumerate(pts):
            cv2.circle(v, (int(px), int(py)), max(4, w // 150), (0, 255, 255), -1)

        plt.figure(figsize=(5, 5))
        plt.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

    return pts


_ = corner_pipeline(make_sample(SCAN_CACHE[SPLIT['test'][0]], np.random.RandomState(5))['photo'], show=True)

## 6. Regularization with dropout

Same architectures, dropout added (bottleneck `Dropout2d` for the encoder–decoders, FC dropout for
the regressor), retrained from scratch and compared.

In [ ]:
enh_drop_model = UNet(p_drop=0.2).to(DEVICE)

if not load_pretrained(enh_drop_model, f'{OUT}/enh_drop_best.pt'):
    enh_drop, enh_drop_hist = train_enhance(UNet(p_drop=0.2), epochs=EPOCHS, tag='enh_drop')
else:
    enh_drop = enh_drop_model
    enh_drop_hist = {'train': [], 'val': []}


reg_drop_model = CornerRegressor(p_drop=0.3).to(DEVICE)

if not load_pretrained(reg_drop_model, f'{OUT}/reg_drop_best.pt'):
    reg_drop, _ = train_corner(CornerRegressor(p_drop=0.3), 'reg', epochs=EPOCHS, tag='reg_drop')
else:
    reg_drop = reg_drop_model


hm_drop_model = CornerHeatmap(p_drop=0.2).to(DEVICE)

if not load_pretrained(hm_drop_model, f'{OUT}/hm_drop_best.pt'):
    hm_drop, _ = train_corner(CornerHeatmap(p_drop=0.2), 'hm', epochs=EPOCHS, tag='hm_drop')
else:
    hm_drop = hm_drop_model

> Training note: The preceding cell loads available dropout checkpoints or trains missing models. Do not retrain unconditionally; set `FORCE_TRAIN = True` in the configuration cell when you explicitly want a fresh training run.


In [ ]:
print('ENHANCEMENT  (PSNR / SSIM)')
for nm, m in [('no dropout', enh_model), ('dropout 0.2', enh_drop)]:
    r = [eval_enh(m, ld)[:2] for ld in (enh_tr_eval, enh_va, enh_te)]
    print(f'  {nm:12s} train {r[0][0]:.2f}/{r[0][1]:.4f}   val {r[1][0]:.2f}/{r[1][1]:.4f}   test {r[2][0]:.2f}/{r[2][1]:.4f}')

print('CORNERS  (mean px err / all-4<8px, synthetic test)')
for nm, m, k in [('reg', reg_model, 'reg'), ('reg+drop', reg_drop, 'reg'), ('hm', hm_model, 'hm'), ('hm+drop', hm_drop, 'hm')]:
    e, a = eval_corner(m, k, cor_te)
    print(f'  {nm:10s}{e:8.2f}px{a:9.2%}')

plt.figure(figsize=(5, 3.4))
plt.plot(enh_hist['val'], label='val, no dropout')
plt.plot(enh_drop_hist['val'], label='val, dropout')
plt.xlabel('epoch')
plt.ylabel('val loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

ENHANCEMENT  (PSNR / SSIM)
  no dropout   train 33.79/0.9751   val 33.58/0.9751   test 33.62/0.9756
  dropout 0.2  train 23.70/0.7683   val 23.62/0.7685   test 24.00/0.7760
CORNERS  (mean px err / all-4<8px, synthetic test)
  reg           3.03px   89.58%
  reg+drop      5.04px   68.75%
  hm           14.63px    0.00%
  hm+drop       0.85px  100.00%

## 7. Bonus — end-to-end scanner

`scan(photo)` = corner pipeline -> `cv2.getPerspectiveTransform` -> `cv2.warpPerspective` ->
enhancement pipeline. Corners are re-ordered TL/TR/BR/BL defensively so a mis-ordered prediction
cannot silently flip the page.

In [ ]:
def order_corners(p):
    p = np.asarray(p, np.float32)
    c = p.mean(0)
    ang = np.arctan2(p[:, 1] - c[1], p[:, 0] - c[0])
    p = p[np.argsort(ang)]
    start = np.argmin(p.sum(1))
    return np.roll(p, -start, axis=0)


def scan(photo_bgr, corners=None, out_size=IMG * 2, show=False):
    if corners is None:
        corners = corner_pipeline(photo_bgr)

    pts = order_corners(corners)
    W = H = out_size

    dst = np.float32([
        [0, 0],
        [W - 1, 0],
        [W - 1, H - 1],
        [0, H - 1]
    ])

    M = cv2.getPerspectiveTransform(pts.astype(np.float32), dst)
    rect = cv2.warpPerspective(photo_bgr, M, (W, H), flags=cv2.INTER_CUBIC)
    out = enhance_pipeline(rect)

    if show:
        v = photo_bgr.copy()
        cv2.polylines(v, [pts.astype(np.int32)], True, (0, 0, 255), 2)

        plt.figure(figsize=(11, 4))

        for i, (im, t) in enumerate([
            (v, 'photo + predicted corners'),
            (rect, 'rectified'),
            (out, 'enhanced')
        ]):
            plt.subplot(1, 3, i + 1)
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            plt.title(t)
            plt.axis('off')

        plt.show()

    return out, rect, pts


for i in range(2):
    s = make_sample(SCAN_CACHE[SPLIT['test'][i % len(SPLIT['test'])]], np.random.RandomState(20 + i))
    _ = scan(s['photo'], show=True)

## Real photos (Sections 1.2, 2.3, 3.3-real, 7)

Put your 10–15 phone photos in `real_photos/`, the Roboflow **COCO keypoints** export at
`real_photos/_annotations.coco.json`, and the commercial-app scans in `real_photos/reference/`
with matching filenames. Every cell below is a no-op if those files are missing, so the notebook
runs end-to-end either way.

In [ ]:
def load_real():
    if not os.path.exists(REAL_JSON):
        print('no annotation file — skipping real-photo evaluation')
        return []

    d = json.load(open(REAL_JSON))
    imgs = {im['id']: im for im in d['images']}
    items = []

    for a in d['annotations']:
        seg = a.get('segmentation')

        if not seg or len(seg) == 0:
            continue

        poly = np.array(seg[0], dtype=np.float32).reshape(-1, 2)

        if len(poly) != 4:
            continue

        pts = poly
        im = imgs[a['image_id']]
        path = os.path.join(REAL_DIR, im['file_name'])

        if not os.path.exists(path):
            continue

        ref = os.path.join(REF_DIR, im['file_name'])
        items.append({
            'path': path,
            'corners': order_corners(pts),
            'ref': ref if os.path.exists(ref) else None
        })

    print(len(items), 'annotated real photos')
    return items


REAL = load_real()

In [ ]:
for it in REAL[:5]:
    img = cv2.imread(it['path'])
    W = H = IMG * 2

    M = cv2.getPerspectiveTransform(
        it['corners'].astype(np.float32),
        np.float32([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]])
    )

    rect = cv2.warpPerspective(img, M, (W, H), flags=cv2.INTER_CUBIC)
    ours = enhance_pipeline(rect)

    plt.figure(figsize=(22, 8))

    for i, (im, t) in enumerate([
        (img, 'original input'),
        (rect, 'rectified input'),
        (ours, 'our model')
    ]):
        plt.subplot(1, 3, i + 1)
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(t)
        plt.axis('off')

    plt.suptitle(os.path.basename(it['path']))
    plt.show()

In [ ]:
if REAL:
    print(f"{'model':12s}{'mean err px(256-scale)':>24s}{'all-4 < 8px':>14s}")

    for nm, m, k in [('regression', reg_model, 'reg'), ('heatmap', hm_model, 'hm')]:
        ds = []

        for it in REAL:
            img = cv2.imread(it['path'])
            h, w = img.shape[:2]

            pred = corner_pipeline(img, m, k)

            d = np.linalg.norm((pred - it['corners']) / np.float32([w, h]) * PHOTO, axis=1)
            ds.append(d)

        ds = np.stack(ds)
        print(f'{nm:12s}{ds.mean():24.2f}{(ds.max(1) < 8).mean():14.2%}')

In [ ]:
def ocr_conf(img_bgr):
    import pytesseract
    d = pytesseract.image_to_data(
        cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
        output_type=pytesseract.Output.DICT
    )
    c = [int(v) for v in d['conf'] if int(v) >= 0]
    words = [t for t, v in zip(d['text'], d['conf']) if t.strip() and int(v) > 60]
    return (np.mean(c) if c else 0.0), len(words)


try:
    import pytesseract
    pytesseract.get_tesseract_version()

    print(f"{'photo':22s}{'input':>16s}{'ours':>16s}{'ours+pred corners':>20s}{'reference':>16s}")

    for it in REAL:
        img = cv2.imread(it['path'])
        W = H = IMG * 2

        M = cv2.getPerspectiveTransform(
            it['corners'].astype(np.float32),
            np.float32([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]])
        )

        rect = cv2.warpPerspective(img, M, (W, H), flags=cv2.INTER_CUBIC)
        ours = enhance_pipeline(rect)
        auto = scan(img)[0]
        ref = cv2.imread(it['ref']) if it['ref'] else None

        def format_result(im):
            if im is None:
                return '   n/a'
            conf, words = ocr_conf(im)
            return f'{conf:5.1f}/{words:3d}'

        print(f"{os.path.basename(it['path']):22s}"
              f"{format_result(rect):>16s}"
              f"{format_result(ours):>16s}"
              f"{format_result(auto):>20s}"
              f"{format_result(ref):>16s}")

    print('\n(columns: mean tesseract confidence / #words with conf>60)')

except Exception as e:
    print('OCR skipped:', e)